# ChatInterface + 대화 횟수 세기

- `gr.ChatInterface`로 대화형 채팅 화면
- `history` 매개변수를 실제로 활용해서 "지금까지 몇 번 대화했는지" 세는 기능을 추가

In [1]:
import gradio as gr

# ChatInterface에 넘기는 함수는 (message, history) 두 개를 받음
#   message : 사용자가 방금 입력한 메시지
#   history : 지금까지의 대화 기록 (현재 메시지는 포함 안 됨)
def echo_bot(message, history):
    turn_count = len(history) // 2 + 1
    # history는 [{"role": "user", "content":...}, {"role": "assistant", "content":...}]
    # 1회 턴이 user+assistant 2개의 항목을 반환하므로 //2

    return f"[{turn_count}번째 대화] 너가 말한 건: '{message}' 이지?"

c:\source\pythonsource\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
demo = gr.ChatInterface(
    fn=echo_bot,
    title="대화 횟수를 세는 에코 챗봇",
    description="입력한 말을 그대로 따라 하면서, 지금이 몇 번째 대화인지도 함께 알려줍니다.",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# uv pip install openai

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def chat_with_gpt(message, history):
    messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content

demo = gr.ChatInterface(fn=chat_with_gpt, title="ChatGPT와 대화하기")
demo.launch()

In [ ]:
# uv pip install transformers torch
# https://aka.ms/vs/17/release/vc_redist.x64.exe 마이크로소프트 VS 최신버전 다운로드 하여 설치
from transformers import pipeline
import gradio as gr

chatbot = pipeline("text-generation", model="microsoft/DialoGPT-medium")

def chat_local(message, history):
    result = chatbot(message, max_length=100)
    return result[0]["generated_text"]

demo = gr.ChatInterface(fn=chat_local, title="로컬 모델 챗봇")
demo.launch()

In [ ]:
import os
import gradio as gr

from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정하세요.")

client = genai.Client(api_key=api_key)


def chat_with_gemini(message, history):
    # Gradio history를 Gemini 형식으로 변환
    gemini_history = []

    for item in history:
        if item["role"] not in ("user", "assistant"):
            continue

        if not isinstance(item["content"], str):
            continue

        gemini_history.append({
            # Gradio의 assistant 역할은 Gemini에서 model
            "role": "model" if item["role"] == "assistant"
            else "user",
            "parts": [
                {"text": item["content"]}
            ]
        })

    chat = client.chats.create(
        model="gemini-3.6-flash",
        history=gemini_history
    )

    response = chat.send_message(message)

    return response.text


demo = gr.ChatInterface(
    fn=chat_with_gemini,
    title="Gemini와 대화하기",
    description="Gemini 3.6 Flash 무료 티어 챗봇입니다."
)

demo.launch()